# Kimi

In [1]:
import os
import json
import tempfile
from pathlib import Path

import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from huggingface_hub import snapshot_download
from kimia_infer.api.kimia import KimiAudio

CACHE_DIR    = "/root/autodl-tmp/LLM_Model"
PROJECT_ROOT = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection")
MODEL_ID     = "moonshotai/Kimi-Audio-7B-Instruct"

os.environ["HF_HOME"] = CACHE_DIR
LOCAL_MODEL_PATH = snapshot_download(MODEL_ID, cache_dir=CACHE_DIR)

Fetching 64 files:   0%|          | 0/64 [00:00<?, ?it/s]

In [2]:
model = KimiAudio(model_path=LOCAL_MODEL_PATH, load_detokenizer=True)
print("Kimi Audio model loaded.")

2026-03-24 09:13:10.888 | INFO     | kimia_infer.api.kimia:__init__:16 - Loading kimi-audio main model
2026-03-24 09:13:10.890 | INFO     | kimia_infer.api.kimia:__init__:25 - Looking for resources in /root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b
2026-03-24 09:13:10.891 | INFO     | kimia_infer.api.kimia:__init__:26 - Loading whisper model
`torch_dtype` is deprecated! Use `dtype` instead!
using normal flash attention


Loading checkpoint shards:   0%|          | 0/36 [00:00<?, ?it/s]

2026-03-24 09:13:20.176 | INFO     | kimia_infer.api.prompt_manager:__init__:20 - Looking for resources in /root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b
2026-03-24 09:13:20.178 | INFO     | kimia_infer.api.prompt_manager:__init__:21 - Loading whisper model
2026-03-24 09:13:21.111 | INFO     | kimia_infer.api.prompt_manager:__init__:30 - Loading text tokenizer
2026-03-24 09:13:21.298 | INFO     | kimia_infer.api.kimia:__init__:41 - Loading detokenizer


ninja: no work to do.


/root/autodl-tmp/envs/kimi/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading '/root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b/vocoder/model.pt'
Complete.
using rope base theta = 10000.0, interpolation factor = 1.0
Currently using bfloat16 for PrefixFlowMatchingDetokenizer
Kimi Audio model loaded.


In [3]:
SYSTEM_PROMPT = (
    "Output one word: Dementia or Control."
)

USER_PROMPT = "Dementia or Control?"

In [4]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="kimi_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [5]:
VALID_LABELS = {"Dementia", "Control"}


def classify_audio(wav_path: Path) -> str:
    """Classify a single audio file. Returns raw model response."""
    messages = [
        {"role": "user", "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT},
        {"role": "user", "message_type": "audio", "content": str(wav_path)},
    ]
    _, text = model.generate(messages, output_type="text")
    return text

In [6]:
def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    predictions, skipped = [], 0

    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]))
            cleaned = raw.strip().strip("'\".,;:!?").capitalize()
            pred = cleaned if cleaned in VALID_LABELS else None
        except Exception as e:
            raw, pred = str(e), None
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred):.4f}")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1):.4f}")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1) :.4f}")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [7]:
import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
from data_split import create_test_csv

csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt", "Pitt", "Pitt_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Pitt"
evaluate_dataset(csv, audio_dir, "Pitt-raw")

[Pitt-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Pitt, exists=True


Pitt-raw:   0%|          | 0/551 [00:00<?, ?it/s]

Pitt-raw:   0%|          | 1/551 [00:02<19:50,  2.17s/it]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-raw:   0%|          | 2/551 [00:02<10:10,  1.11s/it]

...

  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-raw:  25%|██▌       | 140/551 [00:46<02:20,  2.92it/s]

  INVALID [139] session=158-0 true=Control raw='dementia'


Pitt-raw: 100%|██████████| 551/551 [03:14<00:00,  2.84it/s]

[Pitt-raw]
  Accuracy:    0.5873
  F1:          0.7301
  Control Acc: 0.0664
  Dementia Acc:0.9935
  Valid: 550/551  Skipped: 0


In [8]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Lu"
evaluate_dataset(csv, audio_dir, "Lu-raw")

[Lu-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Lu, exists=True


Lu-raw:   1%|▏         | 1/74 [00:00<00:23,  3.05it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-raw:   3%|▎         | 2/74 [00:00<00:21,  3.41it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


...

  INVALID [12] session=F35_004 true=Control raw='dementia'


Lu-raw:  32%|███▏      | 24/74 [00:06<00:14,  3.57it/s]

  INVALID [23] session=F45_000 true=Control raw='dementia'


Lu-raw: 100%|██████████| 74/74 [00:20<00:00,  3.61it/s]

[Lu-raw]
  Accuracy:    0.5211
  F1:          0.6731
  Control Acc: 0.0606
  Dementia Acc:0.9211
  Valid: 71/74  Skipped: 0


In [9]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Demucs"
evaluate_dataset(csv, audio_dir, "Pitt-Demucs")

[Pitt-Demucs] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Demucs, exists=True


Pitt-Demucs:   0%|          | 1/551 [00:00<03:17,  2.78it/s]

  DEBUG [0] session=002-0 raw='Control.' pred=Control


Pitt-Demucs:   0%|          | 2/551 [00:00<03:19,  2.75it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


...

  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-Demucs:  92%|█████████▏| 508/551 [03:00<00:23,  1.81it/s]

  INVALID [507] session=579-0 true=Dementia raw='A woman doing dishes, a boy climbing up to get some cookies, a girl waiting to get some of the cookies, a bench is falling over with the boy, the water is dripping out on the floor.'


Pitt-Demucs: 100%|██████████| 551/551 [03:13<00:00,  2.84it/s]

[Pitt-Demucs]
  Accuracy:    0.5818
  F1:          0.7209
  Control Acc: 0.0950
  Dementia Acc:0.9643
  Valid: 550/551  Skipped: 0


In [10]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Demucs"
evaluate_dataset(csv, audio_dir, "Lu-Demucs")

[Lu-Demucs] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Demucs, exists=True


Lu-Demucs:   1%|▏         | 1/74 [00:00<00:24,  2.93it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-Demucs:   3%|▎         | 2/74 [00:00<00:20,  3.43it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


...

  INVALID [16] session=F38_000 true=Control raw='dementia'


Lu-Demucs:  76%|███████▌  | 56/74 [00:14<00:04,  3.74it/s]

  INVALID [55] session=F17_000 true=Dementia raw='dementia'


Lu-Demucs: 100%|██████████| 74/74 [00:19<00:00,  3.84it/s]

[Lu-Demucs]
  Accuracy:    0.5417
  F1:          0.6916
  Control Acc: 0.0571
  Dementia Acc:1.0000
  Valid: 72/74  Skipped: 0


In [11]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Denoiser"
evaluate_dataset(csv, audio_dir, "Pitt-Denoiser")

[Pitt-Denoiser] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Denoiser, exists=True


Pitt-Denoiser:   0%|          | 1/551 [00:00<03:07,  2.93it/s]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-Denoiser:   0%|          | 2/551 [00:00<03:03,  2.99it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


Pitt-Denoiser:   1%|          | 3/551 [00:01<03:05,  2.96it/s]

  DEBUG [2] session=002-2 raw='Dementia.' pred=Dementia


Pitt-Denoiser: 100%|██████████| 551/551 [02:50<00:00,  3.24it/s]

[Pitt-Denoiser]
  Accuracy:    0.5826
  F1:          0.7222
  Control Acc: 0.0909
  Dementia Acc:0.9676
  Valid: 551/551  Skipped: 0


In [12]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Denoiser"
evaluate_dataset(csv, audio_dir, "Lu-Denoiser")

[Lu-Denoiser] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Denoiser, exists=True


Lu-Denoiser:   1%|▏         | 1/74 [00:00<00:19,  3.73it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-Denoiser:   3%|▎         | 2/74 [00:00<00:17,  4.02it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


...

  INVALID [31] session=F52_000 true=Control raw='dementia'


Lu-Denoiser:  76%|███████▌  | 56/74 [00:13<00:04,  4.14it/s]

  INVALID [55] session=F17_000 true=Dementia raw='dementia'


Lu-Denoiser: 100%|██████████| 74/74 [00:17<00:00,  4.26it/s]

[Lu-Denoiser]
  Accuracy:    0.5588
  F1:          0.7115
  Control Acc: 0.0323
  Dementia Acc:1.0000
  Valid: 68/74  Skipped: 0


In [13]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Pitt-FRCRN_SE")

[Pitt-FRCRN_SE] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-FRCRN_SE, exists=True


Pitt-FRCRN_SE:   0%|          | 1/551 [00:00<02:56,  3.12it/s]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-FRCRN_SE:   0%|          | 2/551 [00:00<03:03,  2.99it/s]

  DEBUG [1] session=002-1 raw='Dementia.' pred=Dementia


Pitt-FRCRN_SE:   1%|          | 3/551 [00:01<03:05,  2.95it/s]

  DEBUG [2] session=002-2 raw='Dementia.' pred=Dementia


Pitt-FRCRN_SE: 100%|██████████| 551/551 [02:50<00:00,  3.23it/s]

[Pitt-FRCRN_SE]
  Accuracy:    0.5771
  F1:          0.7203
  Control Acc: 0.0744
  Dementia Acc:0.9709
  Valid: 551/551  Skipped: 0


In [14]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Lu-FRCRN_SE")

[Lu-FRCRN_SE] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-FRCRN_SE, exists=True


Lu-FRCRN_SE:   1%|▏         | 1/74 [00:00<00:19,  3.74it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-FRCRN_SE:   3%|▎         | 2/74 [00:00<00:17,  4.04it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


...

  INVALID [29] session=F50_000 true=Control raw='dementia'


Lu-FRCRN_SE:  47%|████▋     | 35/74 [00:08<00:08,  4.53it/s]

  INVALID [34] session=F54_000 true=Control raw='dementia'


Lu-FRCRN_SE: 100%|██████████| 74/74 [00:17<00:00,  4.23it/s]

[Lu-FRCRN_SE]
  Accuracy:    0.6269
  F1:          0.7525
  Control Acc: 0.1379
  Dementia Acc:1.0000
  Valid: 67/74  Skipped: 0


In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-MossFormer"
evaluate_dataset(csv, audio_dir, "Pitt-MossFormer")

[Pitt-MossFormer] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-MossFormer, exists=True


Pitt-MossFormer:   0%|          | 1/551 [00:00<02:45,  3.31it/s]

  DEBUG [0] session=002-0 raw='Control' pred=Control


Pitt-MossFormer:   0%|          | 2/551 [00:00<03:00,  3.05it/s]

  DEBUG [1] session=002-1 raw='Dementia.' pred=Dementia


...

Pitt-MossFormer:  25%|██▌       | 140/551 [00:40<02:05,  3.27it/s]

  INVALID [139] session=158-0 true=Control raw='dementia'


Pitt-MossFormer:  62%|██████▏   | 340/551 [01:43<01:22,  2.56it/s]

  INVALID [339] session=157-2 true=Dementia raw='Dementia or Control?'


Pitt-MossFormer:  79%|███████▊  | 433/551 [02:15<00:33,  3.52it/s]

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-MossFormer"
evaluate_dataset(csv, audio_dir, "Lu-MossFormer")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Resemble"
evaluate_dataset(csv, audio_dir, "Pitt-Resemble")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Resemble"
evaluate_dataset(csv, audio_dir, "Lu-Resemble")